In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.dim_customer AS
SELECT
    customerkey,
    gender,
    continent,
    country,
    state,
    city
FROM electronics_cat.silver.customers;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.dim_product AS
SELECT
    productkey,
    product_name,
    brand,
    color,
    category,
    subcategory,
    unit_price_usd
FROM electronics_cat.silver.products;

CREATE OR REPLACE TABLE electronics_cat.gold.dim_store AS
SELECT
    store_key,
    country AS store_country,
    state,
    square_meters
FROM electronics_cat.silver.stores;

CREATE OR REPLACE TABLE electronics_cat.gold.dim_date AS
SELECT DISTINCT
    order_date AS date,
    YEAR(order_date) AS year,
    MONTH(order_date) AS month,
    DAY(order_date) AS day
FROM electronics_cat.silver.sales
WHERE order_date IS NOT NULL;

CREATE OR REPLACE TABLE electronics_cat.gold.dim_exchange_rate AS
SELECT
    date,
    currency,
    exchange
FROM electronics_cat.silver.exc_rate;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.fact_sales AS
SELECT
    s.order_number,
    s.line_item,

    s.order_date,
    d.year,
    d.month,

    s.customerkey,
    s.product_key,
    s.storekey,

    s.quantity,

    p.unit_price_usd,

    er.exchange,

    -------------------------------------------------------------
    ROUND(
        (s.quantity * p.unit_price_usd) / er.exchange,
        2
    ) AS revenue_usd,

    -------------------------------------------------------------
    DATEDIFF(s.delivery_date, s.order_date) AS delivery_days,

    -------------------------------------------------------------
    CASE 
        WHEN s.storekey = -1 THEN 'online'
        ELSE 'store'
    END AS channel,

    s.currency_code

FROM electronics_cat.silver.sales s
LEFT JOIN electronics_cat.gold.dim_product p
    ON s.product_key = p.productkey
LEFT JOIN electronics_cat.gold.dim_date d
    ON s.order_date = d.date
LEFT JOIN electronics_cat.gold.dim_exchange_rate er
    ON s.currency_code = er.currency
    AND s.order_date = er.date;

In [0]:
SELECT
    f.year,
    f.month,
    ROUND(SUM(f.revenue_usd),2) AS revenue_usd
FROM gold.fact_sales f
WHERE f.year = 2024
GROUP BY f.year, f.month
ORDER BY f.month;

In [0]:
WITH monthly AS (
    SELECT month, SUM(revenue_usd) AS revenue
    FROM gold.fact_sales
    WHERE year = 2024
    GROUP BY month
),
total AS (
    SELECT SUM(revenue) AS total_rev FROM monthly
)
SELECT
    m.month,
    ROUND(m.revenue,2),
    ROUND(m.revenue * 100 / t.total_rev,2) AS percent
FROM monthly m, total t
ORDER BY m.revenue DESC
LIMIT 3;

In [0]:
SELECT
    p.category,
    ROUND(SUM(f.revenue_usd),2) AS revenue,
    ROUND(
        SUM(f.revenue_usd)*100 / SUM(SUM(f.revenue_usd)) OVER(),2
    ) AS percent
FROM gold.fact_sales f
JOIN gold.dim_product p
    ON f.productkey = p.productkey
WHERE f.year = 2024
GROUP BY p.category
ORDER BY revenue DESC
LIMIT 3;

In [0]:
SELECT
    ROUND(AVG(delivery_days),2),
    COUNT(*) 
FROM gold.fact_sales
WHERE delivery_days IS NOT NULL;

In [0]:
SELECT
    s.store_country,
    ROUND(AVG(f.delivery_days),2),
    COUNT(*),
    PERCENTILE(f.delivery_days,0.5)
FROM gold.fact_sales f
JOIN gold.dim_store s
    ON f.storekey = s.storekey
GROUP BY s.store_country
ORDER BY AVG(f.delivery_days) DESC
LIMIT 5;